In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Data Analysis & Reporting (Pandas, NumPy, scikit-learn)\\n",
        "\\n",
        "Professional notebook template: profiling, data quality checks, baseline model, and a Markdown report."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import numpy as np\\n",
        "import pandas as pd\\n",
        "\\n",
        "from sklearn.compose import ColumnTransformer\\n",
        "from sklearn.impute import SimpleImputer\\n",
        "from sklearn.model_selection import train_test_split\\n",
        "from sklearn.pipeline import Pipeline\\n",
        "from sklearn.preprocessing import OneHotEncoder, StandardScaler\\n",
        "from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score\\n",
        "from sklearn.linear_model import LogisticRegression, Ridge\\n",
        "\\n",
        "RANDOM_STATE = 42"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Load Data"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "DATA_PATH = r\"data/your_file.csv\"  # change\\n",
        "TARGET = \"target\"  # change\\n",
        "\\n",
        "df = pd.read_csv(DATA_PATH)\\n",
        "df.head()"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Data Quality"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "profile = {\\n",
        "    \"rows\": int(df.shape[0]),\\n",
        "    \"cols\": int(df.shape[1]),\\n",
        "    \"dtypes\": df.dtypes.astype(str).to_dict(),\\n",
        "    \"missing_rate\": df.isna().mean().sort_values(ascending=False).to_dict(),\\n",
        "}\\n",
        "\\n",
        "pd.Series(profile[\"missing_rate\"]).head(10)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Train/Test Split"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "X = df.drop(columns=[TARGET])\\n",
        "y = df[TARGET]\\n",
        "\\n",
        "problem = \"classification\" if (y.dtype == \"O\" or y.nunique(dropna=True) <= 20) else \"regression\"\\n",
        "problem"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "stratify = y if problem == \"classification\" else None\\n",
        "X_train, X_test, y_train, y_test = train_test_split(\\n",
        "    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify\\n",
        ")\\n",
        "\\n",
        "num_cols = [c for c in X_train.columns if pd.api.types.is_numeric_dtype(X_train[c])]\\n",
        "cat_cols = [c for c in X_train.columns if c not in num_cols]\\n",
        "\\n",
        "preprocess = ColumnTransformer(\\n",
        "    transformers=[\\n",
        "        (\"num\", Pipeline([\\n",
        "            (\"imputer\", SimpleImputer(strategy=\"median\")),\\n",
        "            (\"scaler\", StandardScaler()),\\n",
        "        ]), num_cols),\\n",
        "        (\"cat\", Pipeline([\\n",
        "            (\"imputer\", SimpleImputer(strategy=\"most_frequent\")),\\n",
        "            (\"onehot\", OneHotEncoder(handle_unknown=\"ignore\", sparse_output=False)),\\n",
        "        ]), cat_cols),\\n",
        "    ],\\n",
        "    remainder=\"drop\",\\n",
        ")\\n",
        "\\n",
        "model = LogisticRegression(max_iter=2000) if problem == \"classification\" else Ridge(alpha=1.0)\\n",
        "pipe = Pipeline([(\"preprocess\", preprocess), (\"model\", model)])\\n",
        "\\n",
        "pipe.fit(X_train, y_train)\\n",
        "pred = pipe.predict(X_test)\\n",
        "\\n",
        "if problem == \"classification\":\\n",
        "    metrics = {\\n",
        "        \"accuracy\": float(accuracy_score(y_test, pred)),\\n",
        "        \"f1_macro\": float(f1_score(y_test, pred, average=\"macro\")),\\n",
        "    }\\n",
        "else:\\n",
        "    metrics = {\\n",
        "        \"mae\": float(mean_absolute_error(y_test, pred)),\\n",
        "        \"rmse\": float(mean_squared_error(y_test, pred, squared=False)),\\n",
        "        \"r2\": float(r2_score(y_test, pred)),\\n",
        "    }\\n",
        "\\n",
        "metrics"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Generate a Markdown Report"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "top_missing = df.isna().mean().sort_values(ascending=False).head(10)\\n",
        "\\n",
        "lines = []\\n",
        "lines.append(\"# Data Analysis & Reporting\")\\n",
        "lines.append(\"\")\\n",
        "lines.append(f\"- Rows: `{df.shape[0]}`\")\\n",
        "lines.append(f\"- Columns: `{df.shape[1]}`\")\\n",
        "lines.append(f\"- Target: `{TARGET}`\")\\n",
        "lines.append(f\"- Problem type: `{problem}`\")\\n",
        "lines.append(\"\")\\n",
        "lines.append(\"## Quality Checks\")\\n",
        "lines.append(\"\")\\n",
        "lines.append(\"Top missing-rate columns:\")\\n",
        "lines.append(\"\")\\n",
        "for col, rate in top_missing.items():\\n",
        "    lines.append(f\"- `{col}`: `{rate:.1%}` missing\")\\n",
        "lines.append(\"\")\\n",
        "lines.append(\"## Model Metrics\")\\n",
        "lines.append(\"\")\\n",
        "for k, v in metrics.items():\\n",
        "    lines.append(f\"- `{k}`: `{v}`\")\\n",
        "lines.append(\"\")\\n",
        "\\n",
        "report_md = \"\\n\".join(lines)\\n",
        "print(report_md[:800])\\n",
        "\\n",
        "# Save\\n",
        "import os\\n",
        "os.makedirs(\"reports\", exist_ok=True)\\n",
        "with open(\"reports/report.md\", \"w\", encoding=\"utf-8\") as f:\\n",
        "    f.write(report_md)\\n",
        "\\n",
        "\"Wrote reports/report.md\""
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": ""
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
